In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [14]:
df = pd.read_csv(r"D:\Predictive_deploy\cleaned_data.csv")
df.head()

,age,gender,job_type,daily_social_media_time,social_platform_preference,number_of_notifications,work_hours_per_day,perceived_productivity_score,actual_productivity_score,stress_level,sleep_hours,screen_time_before_sleep,breaks_during_work,uses_focus_apps,has_digital_wellbeing_enabled,coffee_consumption_per_day,days_feeling_burnout_per_month,weekly_offline_hours,job_satisfaction_score
0,56,Male,Unemployed,4.180940,Facebook,61,6.753558,8.040464,7.291555,4.0,5.116546,0.419102,8,False,False,4,11,21.927072,6.336688
1,46,Male,Health,3.249603,Twitter,59,9.169296,5.063368,5.165093,7.0,5.103897,0.671519,7,True,True,2,25,0.000000,3.412427
2,32,Male,Finance,3.025913,Twitter,57,7.910952,3.861762,3.474053,4.0,8.583222,0.624378,0,True,False,3,17,10.322044,2.474944
3,60,Female,Unemployed,3.025913,Facebook,59,6.355027,2.916331,1.774869,6.0,6.052984,1.204540,1,False,False,0,4,23.876616,1.733670
4,25,Male,IT,3.025913,Telegram,66,6.214096,8.868753,4.951742,7.0,5.405706,1.876254,1,False,True,1,30,10.653519,9.693060


In [15]:
# Productive label (using threshold of 7.0 — adjustable)
if 'actual_productivity_score' in df.columns:
    df['is_productive'] = (df['actual_productivity_score'] >= 7.0).astype(int)


# Health label based on multiple factors
health_conditions = []
if 'stress_level' in df.columns:
    health_conditions.append(df['stress_level'] <= 3)
if 'sleep_hours' in df.columns:
    health_conditions.append(df['sleep_hours'] >= 7)
if 'days_feeling_burnout_per_month' in df.columns:
    health_conditions.append(df['days_feeling_burnout_per_month'] <= 7)


df['is_healthy'] = (np.column_stack(health_conditions).sum(axis=1) >= 2).astype(int)



print(df[['is_productive', 'is_healthy']].head())
print(df[['is_productive','is_healthy']].value_counts())

   is_productive  is_healthy
0              1           0
1              0           0
2              0           0
3              0           0
4              0           0
is_productive  is_healthy
0              0             20113
               1              5092
1              0              3812
               1               983
Name: count, dtype: int64


In [16]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 25, 35, 50, 80],
    labels=['18-25', '26-35', '36-50', '50+'],
    include_lowest=True
)

In [17]:

le = LabelEncoder()
df['age_group_label'] = le.fit_transform(df['age_group'])

In [18]:
# Ratio Features 
if 'daily_social_media_time' in df.columns and 'work_hours_per_day' in df.columns: 
    df['sm_to_work_ratio'] = df['daily_social_media_time'] / (df['work_hours_per_day'] * 60 + 1)

In [19]:
if 'screen_time_before_sleep' in df.columns and 'sleep_hours' in df.columns: 
    df['sleep_screen_ratio'] = df['screen_time_before_sleep'] / (df['sleep_hours'] + 1)

In [20]:
categorical_cols = df.select_dtypes(include=['object', 'bool', 'category']).columns
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [21]:
# Remove any constant columns
df = df.loc[:, df.nunique() > 1]

In [22]:
df.drop([
    "actual_productivity_score",
    "perceived_productivity_score"
], axis=1, inplace=True)


In [25]:
df.to_csv(r"D:\Predictive_deploy\model_ready.csv", index=False)
print(f"Model-ready dataset saved")

Model-ready dataset saved
